# Lab 5: Debugging the Compilation Stack

**Course**: *Dataflow Architectures for AI: From Principles to Practice*  
**Week**: 5, Debugging Across the Dataflow Stack  
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research)  
**Lab Instructor**: Dr. Rashed Z. Bhatti (IBM Research)  
**Runtime**: Google Colab, GPU runtime required (T4)  
**Expected time**: 90-120 minutes  

---

## Learning objectives
By the end of this lab you will be able to:
1. Detect and fix graph breaks with `torch._dynamo.explain()` and `TORCH_LOGS`.
2. Quantify numerical divergence between eager and compiled execution.
3. Apply bisection across compilation stages to isolate where a numerical change enters.
4. Read the compiler's own artifacts (FX graph, generated kernels) to see what it did.

## Prerequisites
Labs 1-4 complete, plus the Week 5 lecture on debugging across the stack,
bisection, and the diagnostic funnel.

## How this Self-Assessment Lab works

**Practice Lab**: You run this lab on **Google Colab** (free T4).
**OPTIONAL SELF GRADING:** **A separate grader notebook is available on Coursera Labs for self-evaluation**.

| | Where | What happens |
|---|---|---|
| **1. This notebook** | Google Colab (T4 GPU) | Complete every `TODO`, make every **Checkpoint** pass, then run the final cell to write **`lab5_results.json`** |
| **2. Grader notebook** | Coursera Labs (no GPU) | For automated self assessment, paste the contents of `lab5_results.json`, then `Run All`. **40 ungraded points** |

The grader notebook has no GPU and no network, so it cannot re-run any
compilation. Instead it checks that your numbers are **internally consistent** -
the graph count must follow from your break count, each speedup from your own
two timings, the first divergent bisection stage from the three per-stage
differences you measured. Those relations hold for any real run on any GPU,
which is why a free-tier T4 earns a full score.

> **Timings are never graded on magnitude.** Colab VMs are shared, and the same
> code can measure 0.8 ms on one run and 1.3 ms on the next. The grader checks
> that each speedup follows from *your* two timings, never that it hit a
> particular value.

> **Students are encouraged to experiment with other GPUs and accelerators to expand their learning.** The free-tier T4 GPU is freely available on Google
> Colab. Re-running on an L4, V100 or A100 is genuinely instructive and you are
> encouraged to try it. There is **no extra credit** for doing so.

**Self-Assessment Lab ungraded points:**

| Part | Points | Comes from |
|---|---:|---|
| A - Structural invariants | 12 | Parts 1-6 of this notebook |
| B - Self-consistency | 20 | Parts 1-5 |
| C - Physics & plausibility | 8 | Parts 1-5 |
| **Ungraded notebook total** | **40** | fully auto-graded from your JSON for Self-Assessment |

**Graded Quiz points:**
A 10 point graded Coursera Assignment quiz is given to assess your understanding
from this lab experience.

| Quiz | Points | Comes from |
|---|---:|---|
| Concept Check quiz | 10 | Auto-graded Coursera Assignment |

The **Checkpoint** cells in this notebook are self-checks: they catch mistakes
as you work and assign no grade. Treat them as a preview - if a Checkpoint fails
here, the corresponding grader cell will fail there too.

### Working through this lab
1. Set **Runtime -> Change runtime type -> T4 GPU** before you run anything.
2. Run each cell in order. Make sure every **Checkpoint** cell passes (no `AssertionError`).
3. Fill in every `TODO`.
4. Run the final cell to produce **`lab5_results.json`**, then open the grader
   notebook on Coursera and paste it in for self-assessment.
5. Take the **Concept Check quiz** - a separate auto-graded Coursera Assignment
   worth **10 graded points**. It is mandatory, and you need 80% to pass.


## Environment Preflight

This cell detects your runtime environment and sets the execution mode:
- **Full GPU**: all exercises run on CUDA (recommended: T4 or better in Colab)
- **Reduced CPU**: GPU-dependent cells use CPU fallback with reduced workloads
- **Static read-only**: no execution; pre-computed outputs are displayed

In [ ]:
# === Environment Preflight ===
import sys
import importlib

# Check Python version
assert sys.version_info >= (3, 9), f"Python >= 3.9 required, got {sys.version}"

# Check PyTorch
try:
    import torch
    _torch_version = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
    assert _torch_version >= (2, 2), f"PyTorch >= 2.2 required, got {torch.__version__}"
except ImportError:
    raise RuntimeError("PyTorch not installed. Run: pip install torch>=2.2")

# Detect execution mode
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODE = "full-gpu"
    _device_name = torch.cuda.get_device_name(0)
else:
    DEVICE = torch.device("cpu")
    MODE = "reduced-cpu"
    _device_name = "CPU only"

print(f"╔══════════════════════════════════════════════════╗")
print(f"║  Environment Preflight                           ║")
print(f"╠══════════════════════════════════════════════════╣")
print(f"║  Python:    {sys.version.split()[0]:<37s}║")
print(f"║  PyTorch:   {torch.__version__:<37s}║")
print(f"║  Device:    {_device_name:<37s}║")
print(f"║  Mode:      {MODE:<37s}║")
print(f"╚══════════════════════════════════════════════════╝")

if MODE == "reduced-cpu":
    print("\n⚠️  No GPU detected. GPU-dependent cells will use CPU fallback.")
    print("   For full experience: Runtime → Change runtime type → T4 GPU")


---
## 0 · Setup

In [ ]:
!pip -q install transformers==4.44.2

import os
import math
import torch
import torch.nn as nn
import torch._dynamo as dynamo
from torch.profiler import profile, ProfilerActivity, record_function
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(42)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch {torch.__version__} | device: {DEV}')
if DEV == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# PyTorch version compatibility check
import torch
print(f'PyTorch version: {torch.__version__}')
assert int(torch.__version__.split('.')[0]) >= 2, (
    f'This lab requires PyTorch 2.x or later. Got {torch.__version__}. '
    f'Run: pip install --upgrade torch'
)
print('✓ PyTorch version compatible')


We use a **deliberately buggy** transformer block that contains common anti-patterns
students encounter when trying to compile real models. Your job is to find and fix them.

In [ ]:
class BuggyTransformerBlock(nn.Module):
    """A transformer block with deliberate compilation anti-patterns.
    
    This block has THREE bugs that affect compilation:
    1. A graph break (silent performance killer)
    2. A numerical precision issue
    3. A dynamic shape that prevents static compilation
    """
    def __init__(self, hidden_dim=512, num_heads=8, ffn_dim=2048):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        
        self.q_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.o_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)
        self.ffn_up = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.ffn_down = nn.Linear(ffn_dim, hidden_dim, bias=False)
        
        # Bug 2 setup: a threshold stored as a Python float that will
        # cause precision-dependent behavior
        self.scale_threshold = 0.5
    
    def forward(self, x):
        B, S, H = x.shape
        
        # --- Attention block ---
        h = self.ln1(x)
        q = self.q_proj(h).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(h).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(h).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=-1)
        
        # BUG 1: .item() call causes a graph break
        # The student must find this and replace it with a pure-tensor operation
        max_attn = attn.max().item()
        if max_attn > self.scale_threshold:
            attn = attn * 0.9  # "attention dampening" - pedagogical anti-pattern
        
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).reshape(B, S, H)
        out = self.o_proj(out)
        x = x + out
        
        # --- FFN block ---
        h = self.ln2(x)
        
        # BUG 3: Dynamic shape from Python control flow
        # This creates a graph break when sequence length varies
        if h.shape[1] > 64:
            h = self.ffn_up(h[:, :64, :])  # truncate long sequences
            h = torch.nn.functional.pad(h, (0, 0, 0, S - 64))  # pad back
        else:
            h = self.ffn_up(h)
        
        # BUG 2: Manual softmax without numerical stability (no max subtraction)
        # This diverges from torch.softmax which subtracts max for stability
        h_exp = torch.exp(h)  # unstable: can overflow in fp16
        h = h_exp / h_exp.sum(dim=-1, keepdim=True)  # manual "softmax" normalization
        
        h = self.ffn_down(h)
        x = x + h
        return x


# Instantiate
HIDDEN_DIM = 512
NUM_HEADS = 8
FFN_DIM = 2048
BATCH = 2
SEQ = 128

model = BuggyTransformerBlock(HIDDEN_DIM, NUM_HEADS, FFN_DIM).to(DEV).eval()
dummy = torch.randn(BATCH, SEQ, HIDDEN_DIM, device=DEV)

# Verify it runs in eager mode
with torch.no_grad():
    out_eager = model(dummy)
print(f'Eager output shape: {out_eager.shape}')
print(f'Eager output mean: {out_eager.mean().item():.6f}')

---
## 1 · ★ TODO: Detect Graph Breaks

Use `torch._dynamo.explain()` to detect how many graph breaks exist in the
`BuggyTransformerBlock`. Report the count, locations, and reasons.

Then try compiling with `torch.compile` and measure the performance vs eager
to confirm the graph breaks are hurting performance.

In [ ]:
# TODO 1.1: Use torch._dynamo.explain() to detect graph breaks
#
# The current API is explain(fn)(*args) -- NOT explain(fn, *args).
# Store your results in these exact names; the export cell reads them:
#   graph_break_count : int   (expl.graph_break_count)
#   graph_count       : int   (expl.graph_count)
#   break_reasons     : list of str, one per break
#
# Print all three. Note how many graphs Dynamo produced for one break.

dynamo.reset()

# YOUR CODE HERE
# expl = ...
# graph_break_count = ...
# graph_count = ...
# break_reasons = ...
raise NotImplementedError


In [ ]:
# TODO 1.2: Compile the buggy model and benchmark eager vs compiled
#
# With graph breaks present, "compiled" can be SLOWER than eager: every break
# means leaving the compiled region and re-entering it.
#
# Store your results in these exact names:
#   eager_ms      : float, median ms per forward, eager
#   compiled_ms   : float, median ms per forward, torch.compile(model)
#   buggy_speedup : float, eager_ms / compiled_ms
#
# A bench_ms(fn, x) helper is provided -- it warms up, synchronizes, and
# returns the median in ms. Use it for both measurements.

from torch.utils import benchmark as torch_bench

def bench_ms(fn, x):
    """Median wall-clock ms per forward. Warms up, then blocked_autorange."""
    with torch.no_grad():
        for _ in range(3):
            fn(x)
    if DEV == "cuda":
        torch.cuda.synchronize()
    t = torch_bench.Timer(stmt="with torch.no_grad(): fn(x)",
                          globals={"fn": fn, "x": x, "torch": torch})
    return t.blocked_autorange(min_run_time=1.0).median * 1000

dynamo.reset()
model_compiled = torch.compile(model)
# Warm-up run to trigger compilation
with torch.no_grad():
    _ = model_compiled(dummy)

# YOUR CODE HERE
# eager_ms = ...
# compiled_ms = ...
# buggy_speedup = ...
raise NotImplementedError


In [ ]:
# --- Checkpoint 1 -------------------------------------------------------
# The buggy block has data-dependent control flow, so Dynamo must split it.
assert graph_break_count >= 1, (
    f"expected at least one graph break in BuggyTransformerBlock, got "
    f"{graph_break_count}. Call dynamo.explain(model)(dummy) -- the current API "
    f"is explain(fn)(*args), not explain(fn, *args).")
# Dynamo emits one more graph than it has breaks: n breaks -> n+1 segments.
assert graph_count == graph_break_count + 1, (
    f"graphs ({graph_count}) should be graph breaks ({graph_break_count}) + 1: "
    f"each break splits one graph into two.")
assert len(break_reasons) >= 1, "every break should come with a reason string"
assert eager_ms > 0 and compiled_ms > 0, "both timings must be positive"
print(f"[OK] {graph_break_count} graph break(s) -> {graph_count} graphs")
print(f"     eager {eager_ms:.3f} ms vs compiled {compiled_ms:.3f} ms "
      f"({buggy_speedup:.2f}x)")


---
## 2 · ★ TODO: Fix the Graph Breaks

Create a `FixedTransformerBlock` that eliminates all graph breaks while
preserving the intended behavior. The three bugs are:

1. **`.item()` + Python if**: replace with a pure tensor operation (e.g., `torch.where`)
2. **Manual unstable softmax**: replace with `torch.softmax` or add max-subtraction
3. **Shape-dependent branching**: remove the branch or use `torch._dynamo.mark_dynamic`

After fixing, verify with `dynamo.explain()` that break_count == 0.

In [ ]:
class FixedTransformerBlock(nn.Module):
    """The same transformer block with all three graph-related bugs fixed.

    TODO 2.1 / 2.2 / 2.3: fix the three bugs in forward().
      Bug 1  .item() + Python `if`      -> a pure-tensor select (torch.where)
      Bug 2  manual exp/sum "softmax"   -> torch.softmax (numerically stable)
      Bug 3  shape-dependent branching  -> drop the branch entirely

    Keep everything else identical: same modules, same order, same residuals.
    The weights are copied from the buggy model, so a correct fix changes the
    numbers only where the bugs were.
    """
    def __init__(self, hidden_dim=512, num_heads=8, ffn_dim=2048):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.q_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.o_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)
        self.ffn_up = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.ffn_down = nn.Linear(ffn_dim, hidden_dim, bias=False)
        self.scale_threshold = 0.5

    def forward(self, x):
        B, S, H = x.shape

        # --- Attention block ---
        h = self.ln1(x)
        q = self.q_proj(h).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(h).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(h).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=-1)

        # TODO 2.1: Fix Bug 1. The buggy version was:
        #     max_attn = attn.max().item()
        #     if max_attn > self.scale_threshold:
        #         attn = attn * 0.9
        # Reading a tensor value into Python forces a host sync and a graph
        # break. Express the same dampening with a pure-tensor operation.
        # YOUR CODE HERE
        raise NotImplementedError

        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).reshape(B, S, H)
        out = self.o_proj(out)
        x = x + out

        # --- FFN block ---
        h = self.ln2(x)

        # TODO 2.2: Fix Bug 3. The buggy version truncated long sequences to 64
        # positions and zero-padded them back, inside a Python `if` on a shape.
        # Run ffn_up on the full sequence instead.
        # YOUR CODE HERE
        raise NotImplementedError

        # TODO 2.3: Fix Bug 2. The buggy version was:
        #     h_exp = torch.exp(h)
        #     h = h_exp / h_exp.sum(dim=-1, keepdim=True)
        # That omits the max-subtraction and overflows in fp16. Use the stable
        # built-in instead.
        # YOUR CODE HERE
        raise NotImplementedError

        h = self.ffn_down(h)
        x = x + h
        return x


# Instantiate the fixed model with the SAME weights as the buggy one
model_fixed = FixedTransformerBlock(HIDDEN_DIM, NUM_HEADS, FFN_DIM).to(DEV).eval()
model_fixed.load_state_dict(model.state_dict())

# Verify no graph breaks remain
dynamo.reset()
expl_fixed        = dynamo.explain(model_fixed)(dummy)
fixed_break_count = int(expl_fixed.graph_break_count)
fixed_graph_count = int(expl_fixed.graph_count)
print(f"fixed model graph breaks: {fixed_break_count}")
print(f"fixed model graphs      : {fixed_graph_count}")


In [ ]:
# 2.3: compile the fixed model and generate reference outputs
dynamo.reset()
model_fixed_compiled = torch.compile(model_fixed)

# Warm-up compiled model (triggers compilation)
with torch.no_grad():
    _ = model_fixed_compiled(dummy)

# Generate all reference outputs needed for numerical comparison
with torch.no_grad():
    out_buggy_eager   = model(dummy)           # buggy model, eager
    out_fixed_eager   = model_fixed(dummy)     # fixed model, eager
    out_fixed_compiled = model_fixed_compiled(dummy)  # fixed model, compiled

print(f"model_fixed_compiled ready on {DEV}")
print(f"out_buggy_eager   shape: {out_buggy_eager.shape}")
print(f"out_fixed_eager   shape: {out_fixed_eager.shape}")
print(f"out_fixed_compiled shape: {out_fixed_compiled.shape}")


In [ ]:
# TODO 2.4: Benchmark the fixed model (compiled) vs the buggy model (compiled)
#
# Store your results in these exact names:
#   fixed_compiled_ms : float, median ms per forward, fixed model compiled
#   fix_speedup       : float, compiled_ms / fixed_compiled_ms
#
# Reuse the bench_ms helper from Part 1.
#
# Note: the SIZE of this speedup varies a lot with VM contention -- 1.05x and
# 1.6x are both normal for the same code. What matters is the direction and
# that your number follows from your own two timings.

# YOUR CODE HERE
# fixed_compiled_ms = ...
# fix_speedup = ...
raise NotImplementedError


In [ ]:
# --- Checkpoint 2 -------------------------------------------------------
# The whole point of Part 2: a fixed block traces as ONE graph.
assert fixed_break_count == 0, (
    f"the fixed model still has {fixed_break_count} graph break(s). All three "
    f"bugs must go: .item() + Python if -> torch.where; the shape-dependent "
    f"branch -> remove it; manual exp/sum -> torch.softmax.")
assert fixed_graph_count == 1, (
    f"a break-free module traces to exactly 1 graph, got {fixed_graph_count}.")
assert fixed_break_count < graph_break_count, (
    "the fixed model should have strictly fewer breaks than the buggy one")
assert fixed_compiled_ms > 0, "fixed_compiled_ms must be positive"
print(f"[OK] fixed model: 0 graph breaks, 1 graph")
print(f"     buggy compiled {compiled_ms:.3f} ms -> fixed compiled "
      f"{fixed_compiled_ms:.3f} ms ({fix_speedup:.2f}x)")


---
## 3 · ★ TODO: Numerical Divergence Analysis

Now compare the **eager** output of the buggy model against the **eager** output of the
fixed model. The fixes you applied changed the numerical behavior (especially Bug 2: the
unstable softmax). Quantify the divergence.

Then compare the fixed model in **eager** vs **compiled** mode to see what level of
divergence is "normal" from compilation alone.

In [ ]:
# TODO 3.1: Compare buggy eager vs fixed eager
#
# Store your results in these exact names:
#   max_abs_diff  : float, (buggy - fixed).abs().max()
#   mean_abs_diff : float, (buggy - fixed).abs().mean()
#   max_rel_diff  : float, max of |diff| / (|fixed| + 1e-8)
#
# The reference outputs out_buggy_eager / out_fixed_eager were computed in the
# 2.3 cell above.

# YOUR CODE HERE
# diff = ...
# max_abs_diff = ...
# mean_abs_diff = ...
# max_rel_diff = ...
raise NotImplementedError


In [ ]:
# TODO 3.2: Compare fixed eager vs fixed compiled
#
# This isolates divergence from COMPILATION alone -- no model change, just
# fusion and reduction reordering. Expect it to be far smaller than 3.1.
#
# Store your results in these exact names:
#   compilation_max_abs  : float
#   compilation_mean_abs : float
#   compilation_max_rel  : float

# YOUR CODE HERE
# d2 = ...
# compilation_max_abs = ...
# compilation_mean_abs = ...
# compilation_max_rel = ...
raise NotImplementedError


In [ ]:
# TODO 3.3: Assemble the comparison table.
#
# Fill each value from the variables you just computed. The grader recomputes
# `ratio` from your two max-abs figures, so it must be that division.

comparison = {
    "buggy_vs_fixed_max_abs":    None,   # REPLACE with max_abs_diff
    "buggy_vs_fixed_mean_abs":   None,   # REPLACE with mean_abs_diff
    "compilation_only_max_abs":  None,   # REPLACE with compilation_max_abs
    "compilation_only_mean_abs": None,   # REPLACE with compilation_mean_abs
    "is_bug2_divergence_larger": None,   # REPLACE: True/False
    "ratio":                     None,   # REPLACE: buggy_vs_fixed / compilation_only
}

print("Divergence comparison:")
for k, v in comparison.items():
    print(f"  {k}: {v}")


In [ ]:
# --- Checkpoint 3 -------------------------------------------------------
# Fixing the model changes the answer by far more than compiling it does.
# That gap is the lab's central measurement: a model bug and compiler noise
# are different magnitudes, and the tolerance you pick has to separate them.
assert max_abs_diff > 0, "buggy and fixed outputs should differ (Bug 3 truncates the FFN)"
assert compilation_max_abs >= 0, "a max abs diff cannot be negative"
assert comparison["is_bug2_divergence_larger"] is True, (
    f"the model-fix divergence ({max_abs_diff:.3e}) should exceed compilation "
    f"noise ({compilation_max_abs:.3e}). If it does not, check that "
    f"out_buggy_eager came from the BUGGY model and out_fixed_eager from the "
    f"FIXED one.")
assert comparison["ratio"] > 1.0, "ratio = buggy_vs_fixed / compilation_only must exceed 1"
print(f"[OK] model-fix divergence {max_abs_diff:.3e} is "
      f"{comparison['ratio']:.0f}x the compilation noise {compilation_max_abs:.3e}")


### Tolerance Selection and False Positives

Choosing `atol` and `rtol` requires understanding:
1. **Dtype precision**: FP32 has ~7 decimal digits; FP16 has ~3.3; BF16 has ~3.3 but with FP32 range
2. **Accumulation order**: reductions (sum, mean, softmax) are non-associative in floating point
3. **Determinism**: some CUDA ops are non-deterministic by default (atomics in backward pass)

| Comparison | Typical `atol` | Typical `rtol` | Rationale |
|---|---|---|---|
| eager vs eager (same seed) | 0 | 0 | Bitwise identical |
| eager vs compiled (FP32) | 1e-5 | 1e-5 | Reordered ops |
| eager vs compiled (FP16) | 1e-2 | 1e-2 | Low precision + reordering |
| across GPU architectures | 1e-3 | 1e-3 | Different rounding hardware |

> ⚠️ **False positive risk:** Setting `atol` too tight flags benign reordering as bugs.
> Setting it too loose hides real numerical issues.

In [ ]:
# Demonstrate false positives from overly tight tolerance
#
# A reordered reduction over many fp32 values does not return the same bits.
# We make the effect large enough to be unambiguous by mixing magnitudes: a
# wide dynamic range is exactly when accumulation order matters most, which is
# also why reductions are where compiled and eager outputs drift apart.
torch.manual_seed(42)
x = torch.randn(128, 768, device=DEV)
x[0, 0] = 1.0e6          # one large term -> the small terms round off
x[0, 1] = -1.0e6         # cancels it, so the true sum is unchanged

sum_fwd = x.sum()
sum_rev = x.flip(0).sum()          # same values, different addition order

tol_sum_fwd  = float(sum_fwd.item())
tol_sum_rev  = float(sum_rev.item())
tol_abs_diff = abs(tol_sum_fwd - tol_sum_rev)
TOL_TIGHT, TOL_LOOSE = 1e-7, 1e-1
tol_pass_tight = bool(torch.allclose(sum_fwd, sum_rev, atol=TOL_TIGHT, rtol=0))
tol_pass_loose = bool(torch.allclose(sum_fwd, sum_rev, atol=TOL_LOOSE, rtol=0))

print("Same data, different sum order:")
print(f"  sum(x)          = {tol_sum_fwd:.6f}")
print(f"  sum(flip(x))    = {tol_sum_rev:.6f}")
print(f"  |difference|    = {tol_abs_diff:.3e}")
print(f"  allclose(atol={TOL_TIGHT:.0e}) passes? {tol_pass_tight}   <- false positive")
print(f"  allclose(atol={TOL_LOOSE:.0e}) passes? {tol_pass_loose}   <- correct verdict")

# Best practice: scale tolerance by dtype
def recommended_tolerance(dtype):
    """Return (atol, rtol) appropriate for comparing compiled vs eager outputs."""
    if dtype == torch.float32:
        return 1e-5, 1e-5
    elif dtype in (torch.float16, torch.bfloat16):
        return 1e-2, 1e-2
    return 1e-5, 1e-5

atol, rtol = recommended_tolerance(x.dtype)
print(f"\nRecommended for {x.dtype}: atol={atol}, rtol={rtol}")
print("\nThe lesson: reordering a reduction is not a bug. A tolerance tight")
print("enough to flag it produces a false positive on every compiled run.")


In [ ]:
# --- Checkpoint 3b ------------------------------------------------------
# Floating-point addition is not associative, so reordering this reduction
# changes the result. Hardware-independent arithmetic: the wide dynamic range
# in x makes the effect far larger than any per-device rounding difference.
assert tol_abs_diff > TOL_TIGHT, (
    f"reordering the sum changed the result by {tol_abs_diff:.3e}, which is "
    f"below the tight tolerance {TOL_TIGHT:.0e}. Expected a clearly larger "
    f"difference -- check that x still contains the +/-1e6 pair, which is what "
    f"forces the small terms to round off.")
assert tol_pass_tight is False, (
    f"atol={TOL_TIGHT:.0e} should FAIL on a reordered sum (diff "
    f"{tol_abs_diff:.3e}) -- that false positive is what this cell demonstrates.")
assert tol_pass_loose is True, (
    f"atol={TOL_LOOSE:.0e} should PASS (diff {tol_abs_diff:.3e}): loose enough "
    f"to tolerate reordering, still far tighter than a real bug would need.")
print(f"[OK] reordered sum differs by {tol_abs_diff:.3e}: "
      f"{TOL_TIGHT:.0e} flags it (false positive), {TOL_LOOSE:.0e} does not")


---
## 4 · ★ TODO: Bisection Debugging

Now practice the **bisection method** from lecture. Compile the *fixed* model
through three progressively deeper backends and compare each against the eager
baseline, to find the first stage at which the numbers change:

| Stage | Backend | What it does |
|---|---|---|
| 1 | `eager` | Dynamo captures a graph, then replays the same kernels |
| 2 | `aot_eager` | AOTAutograd decomposes the graph, still dispatching ATen kernels |
| 3 | `inductor` | Full codegen: fusion, reduction reordering, generated Triton kernels |

Stages 1 and 2 do not change the arithmetic, so they reproduce eager
**exactly**. Stage 3 does change it, so expect a small nonzero difference.

> **What counts as "changed"?** Not "bigger than some constant I picked". The
> cell below first runs eager **against itself** to establish how much this
> forward pass varies when nothing has changed — on a deterministic pass, that
> is exactly `0.0`. A stage then counts as divergent when it differs from eager
> by more than eager differs from itself. That comparison needs no invented
> tolerance and gives the same verdict on every device.
>
> This matters because the Inductor effect here is **a couple of fp32 ulps** —
> about `2⁻²³` relative to the output. Any "reasonable-looking" absolute
> threshold such as `1e-6` sits *above* it and would report, wrongly, that
> nothing diverged at all. The size of a floating-point difference only means
> something relative to the scale of the data and to the reproducibility of the
> thing you are comparing against.


In [ ]:
# Bisection setup: compare outputs at each compilation stage
dynamo.reset()

# Stage 0: Eager baseline (ground truth).
with torch.no_grad():
    baseline = model_fixed(dummy)

# Eager's own reproducibility is the reference we compare the stages against.
# Re-running the same module on the same input returns the same bits, so this
# is 0.0 -- and that makes it the right yardstick: a stage "changed the
# arithmetic" when it differs from eager MORE than eager differs from itself.
with torch.no_grad():
    baseline_again = model_fixed(dummy)
eager_self_diff = (baseline - baseline_again).abs().max().item()

# Stage 1: Graph capture only (no optimization) -- "eager" backend
dynamo.reset()
model_stage1 = torch.compile(model_fixed, backend="eager")
with torch.no_grad():
    out_stage1 = model_stage1(dummy)

# Stage 2: AOT autograd decomposition (no codegen) -- "aot_eager" backend
dynamo.reset()
model_stage2 = torch.compile(model_fixed, backend="aot_eager")
with torch.no_grad():
    out_stage2 = model_stage2(dummy)

# Stage 3: Full compilation with the Inductor backend
dynamo.reset()
model_stage3 = torch.compile(model_fixed, backend="inductor")
with torch.no_grad():
    out_stage3 = model_stage3(dummy)

stage_diffs = {
    1: (baseline - out_stage1).abs().max().item(),
    2: (baseline - out_stage2).abs().max().item(),
    3: (baseline - out_stage3).abs().max().item(),
}

# Context for reading those numbers: one fp32 ulp at this output magnitude.
# This is reported, not used as a threshold -- see the note below the table.
FP32_EPS  = 2.0 ** -23                      # ~1.19e-07
out_scale = baseline.abs().max().item()
ULP       = out_scale * FP32_EPS

print("Bisection results (max abs diff from the eager baseline):")
print(f"  eager vs itself                : {eager_self_diff:.3e}   <- the yardstick")
for _s, _label in [(1, "graph capture only"), (2, "AOT decomposition"),
                   (3, "full Inductor")]:
    _d = stage_diffs[_s]
    _verdict = "same arithmetic" if _d <= eager_self_diff else "CHANGED the arithmetic"
    print(f"  Stage {_s} ({_label:19s}): {_d:.3e}   {_verdict}")
print()
print(f"  output magnitude : {out_scale:.4f}")
print(f"  1 fp32 ulp here  : {ULP:.3e}")
if stage_diffs[3] > 0:
    print(f"  stage 3 is {stage_diffs[3]/ULP:.2f} ulp -- a real difference, and a tiny one")


In [ ]:
# TODO 4.1: Interpret the bisection.
#
# Derive `first_divergent_stage` from the measurements -- do not type in a
# number. The cell above computed `stage_diffs` and `eager_self_diff`: a stage
# changed the arithmetic when it differs from eager by MORE than eager differs
# from itself.
#
# Hint: next((s for s in (1, 2, 3) if <condition>), None)

# YOUR CODE HERE
# first_divergent_stage = ...
# stage3_ulps = ...   # stage_diffs[3] / ULP, for interpretation

bisection_result = {
    "stage1_max_abs":        stage_diffs[1],
    "stage2_max_abs":        stage_diffs[2],
    "stage3_max_abs":        stage_diffs[3],
    "eager_self_diff":       eager_self_diff,
    "output_scale":          out_scale,
    "fp32_ulp":              ULP,
    "stage3_ulps":           None,   # REPLACE with stage3_ulps
    "first_divergent_stage": None,   # REPLACE with first_divergent_stage
    "divergence_cause":      None,   # REPLACE: one or two sentences, your own words
    "is_this_a_bug":         None,   # REPLACE: True/False -- is this divergence a bug?
}

print("Bisection interpretation:")
for k, v in bisection_result.items():
    print(f"  {k}: {v}")


In [ ]:
# --- Checkpoint 4 -------------------------------------------------------
# Bisection localises the change: capture and decomposition preserve the
# numerics exactly, codegen perturbs them slightly. What is graded is the
# ORDERING of the stages, never the size of the perturbation -- that size is a
# property of the hardware and the Inductor version, and on this model it is
# only a couple of ulps.
assert eager_self_diff == 0.0, (
    f"eager differs from itself by {eager_self_diff:.3e}. This forward pass "
    f"should be deterministic; if it is not, the bisection has no stable "
    f"baseline to compare against.")
assert stage_diffs[1] == 0.0, (
    f"stage 1 (graph capture) differs from eager by {stage_diffs[1]:.3e}. The "
    f"'eager' backend only captures the graph and replays the same kernels, so "
    f"it should match bit for bit.")
assert stage_diffs[2] == 0.0, (
    f"stage 2 (AOT decomposition) differs from eager by {stage_diffs[2]:.3e}. "
    f"aot_eager decomposes the graph but still dispatches the same kernels.")
assert stage_diffs[3] > 0.0, (
    f"stage 3 (Inductor) matched eager exactly. Expected a small nonzero "
    f"difference from fusion and reduction reordering. If it really is 0, "
    f"check that the Inductor run recompiled (dynamo.reset() first).")
assert first_divergent_stage == 3, (
    f"expected Inductor (stage 3) to be the first divergent stage, got "
    f"{first_divergent_stage}: stages 1-2 must be exactly 0.0 and stage 3 "
    f"nonzero.")
assert bisection_result["is_this_a_bug"] is False, (
    "a couple of ulps from reduction reordering under fusion is expected "
    "floating-point behaviour, not a correctness bug -- that distinction is "
    "the point of Part 4.")
print(f"[OK] eager is reproducible (0.0); stages 1-2 match it exactly")
print(f"     stage 3 diverges by {stage_diffs[3]:.3e} "
      f"({stage3_ulps:.2f} ulp) -- real, and tiny")
print(f"     first divergent stage = {first_divergent_stage} (Inductor codegen)")


---
## 5 · ★ TODO: TORCH_COMPILE_DEBUG Inspection

Use `TORCH_COMPILE_DEBUG` to dump the compilation artifacts and inspect what the
compiler actually did to your fixed model.

In [ ]:
# Run compilation with debug output
#
# WHY THIS SETS CONFIG RATHER THAN ENV VARS
# -----------------------------------------
# TORCH_COMPILE_DEBUG is read exactly once, when torch._inductor.config is
# imported:
#
#     enabled = os.environ.get("TORCH_COMPILE_DEBUG", "0") == "1"
#
# We imported torch back in the setup cell, so that line already ran and the
# flag is already False. Setting the environment variable now is too late and
# silently produces no artifacts at all. The reliable route in a notebook --
# where you cannot set the variable before the import -- is to assign the
# config attributes directly.
import shutil

debug_dir = '/tmp/torch_compile_debug'
if os.path.exists(debug_dir):
    shutil.rmtree(debug_dir)
os.makedirs(debug_dir, exist_ok=True)

_trace = torch._inductor.config.trace
_prev  = (_trace.enabled, _trace.debug_dir)

_trace.enabled   = True        # the master switch for every artifact below
_trace.debug_dir = debug_dir
_trace.fx_graph  = True        # input FX graph (post-decomp, pre-optimization)
_trace.fx_graph_transformed = True
_trace.output_code = True      # the generated Triton / extern kernel code

# Inductor caches compiled graphs on disk, and Part 4 already compiled this
# module with the inductor backend. A cache hit skips codegen, so there would
# be nothing to dump. Disable the caches and compile a fresh instance.
try:
    torch._inductor.config.force_disable_caches = True
except Exception:
    pass

model_debug_src = FixedTransformerBlock(HIDDEN_DIM, NUM_HEADS, FFN_DIM).to(DEV).eval()
model_debug_src.load_state_dict(model_fixed.state_dict())

dynamo.reset()
model_debug = torch.compile(model_debug_src, backend="inductor")
with torch.no_grad():
    _ = model_debug(dummy)

# Restore, so later cells and re-runs are unaffected.
_trace.enabled, _trace.debug_dir = _prev
try:
    torch._inductor.config.force_disable_caches = False
except Exception:
    pass

# List what was produced. Inductor writes into a per-compile subdirectory, so
# search recursively rather than listing debug_dir itself.
import glob
artifacts = [a for a in glob.glob(f'{debug_dir}/**/*', recursive=True)
             if os.path.isfile(a)]
print(f"Debug artifacts produced: {len(artifacts)} files")
for a in sorted(artifacts)[:25]:
    print(f"  {a.replace(debug_dir, '.')}")

if not artifacts:
    print("\n[!] Still no artifacts. Inductor's trace output can also land in a")
    print("    system temp dir when debug_dir is ignored. Try:")
    print("      import glob, tempfile")
    print("      glob.glob(tempfile.gettempdir() + '/torchinductor*/**/output_code.py',")
    print("                recursive=True)")


In [ ]:
# TODO 5.1: Read and inspect the captured FX graph
#
# Find the fx_graph_readable.py the cell above wrote, print its first ~60
# lines, and count what is in it.
#
# Store your results in these exact names:
#   fx_n_ops    : int, approximate operator count (count " = torch.ops.")
#   fx_n_matmul : int, matmul-family ops (aten.mm + aten.bmm + aten.addmm)
#
# A _find(patterns) helper is defined in the cell above; it searches the debug
# dir and falls back to temp dirs. Use _find(["*fx_graph_readable*", ...]).

# YOUR CODE HERE
# fx_files = ...
# fx_n_ops = ...
# fx_n_matmul = ...
raise NotImplementedError


In [ ]:
# TODO 5.2: Read the generated kernel code (output_code.py)
#
# Store your results in these exact names:
#   triton_kernels : int, count of "def triton_"      (kernels Inductor GENERATED)
#   extern_kernels : int, count of "extern_kernels."   (calls OUT to cuBLAS)
#
# Then look at which operators ended up fused together. The split you find is
# the Part 5 lesson: "compiled" does not mean every kernel was written from
# scratch -- the GEMMs are still cuBLAS.

# YOUR CODE HERE
# oc = ...
# triton_kernels = ...
# extern_kernels = ...
raise NotImplementedError


In [ ]:
# --- Checkpoint 5 -------------------------------------------------------
# The Part 5 lesson: Inductor fuses the elementwise/normalisation work into
# generated Triton kernels but hands the GEMMs to cuBLAS. "Compiled" does not
# mean every kernel was written from scratch.
#
# These artifacts depend on the debug dump actually happening, which depends on
# Inductor not serving the compile from cache. That is an environment quirk
# rather than a mistake in your analysis, so a missing dump reports loudly here
# instead of failing the whole notebook -- but the grader does check the counts,
# so do re-run if you see the warning.
if not artifacts:
    print("[!] SKIPPED: no debug artifacts were written (Inductor cache hit).")
    print("    Restart the runtime and Run All to populate them.")
    print("    The exported JSON will carry zeros for this part.")
else:
    assert fx_n_ops > 0, (
        f"artifacts were written but no operators were counted in the FX graph. "
        f"Check which file 5.1 picked up: {fx_files[:1]}")
    assert fx_n_matmul >= 1, (
        f"expected at least one matmul-family op in the graph, found "
        f"{fx_n_matmul}. The block has q/k/v/o projections and two FFN layers.")
    if DEV == "cuda":
        assert triton_kernels >= 1, (
            f"expected Inductor to generate at least one Triton kernel, got "
            f"{triton_kernels}. On CUDA the elementwise and LayerNorm work is fused.")
        assert extern_kernels >= 1, (
            f"expected at least one extern (cuBLAS) GEMM call, got "
            f"{extern_kernels}. Inductor does not generate its own GEMMs by default.")
        print(f"[OK] {fx_n_ops} ops ({fx_n_matmul} matmul-family) -> "
              f"{triton_kernels} Triton + {extern_kernels} extern kernels")
    else:
        print(f"[OK] {fx_n_ops} ops ({fx_n_matmul} matmul-family) captured "
              f"(CPU mode: no Triton/cuBLAS split)")


---

## 6 - Update the Accelerator Landscape Map

Add this week's row to the shared map you have been building since Lab 1.
Week 5's columns are about **debuggability**: what the stack let you see, and
what it would hide on a target with no eager fallback.


In [ ]:
# 6.1: Landscape Map row -- debuggability of the stack you exercised this week.
# Fill the measured values from your own run.
landscape_row_w5 = {
    "accelerator":              (f"GPU ({torch.cuda.get_device_name(0)})"
                                 if torch.cuda.is_available() else "CPU"),
    "debuggability":            "eager fallback hides graph breaks; intermediate values inspectable",
    "graph_breaks_found":       None,   # TODO: int(graph_break_count)
    "graph_breaks_after_fix":   None,   # TODO: int(fixed_break_count)
    "compilation_noise_maxabs": None,   # TODO: round(float(compilation_max_abs), 8)
    "first_divergent_stage":    None,   # TODO: int(first_divergent_stage)
    "spyre_prediction":         "no eager fallback: a graph break becomes a compile-time rejection",
}
for _k, _v in landscape_row_w5.items():
    print(f"  {_k:26s}: {_v}")


In [ ]:
# --- Checkpoint 6 -------------------------------------------------------
_required = {"accelerator", "debuggability", "graph_breaks_found",
             "graph_breaks_after_fix", "compilation_noise_maxabs",
             "first_divergent_stage", "spyre_prediction"}
assert set(landscape_row_w5) == _required, (
    f"landscape_row_w5 keys are wrong: "
    f"{sorted(set(landscape_row_w5) ^ _required)}")
assert all(v is not None for v in landscape_row_w5.values()), (
    "every landscape_row_w5 field must be filled in")

# Each measured field must mirror what you actually measured. The grader
# re-checks every one of these against the raw values in your JSON.
assert landscape_row_w5["graph_breaks_found"] == graph_break_count
assert landscape_row_w5["graph_breaks_after_fix"] == fixed_break_count == 0
assert abs(landscape_row_w5["compilation_noise_maxabs"]
           - compilation_max_abs) < 1e-7, (
    "compilation_noise_maxabs should be your Part 3.2 figure, not the Part 3.1 "
    "model-fix divergence")
assert landscape_row_w5["first_divergent_stage"] == first_divergent_stage
print("[OK] Week 5 Landscape Map row complete and consistent with your measurements")


---

## Check your understanding (ungraded)

These prompts are **not graded and not submitted** - nothing here travels in
your JSON, and no one reviews your answers. Work through them anyway: they are
the reasoning the **Concept Check quiz** tests, and that quiz *is* the graded
one, worth **10 graded points**. Answer in your own words, citing the numbers
*you* measured.

---

### 1. Why did compiling the buggy model make it slower?

You measured `eager_ms` and `compiled_ms` for a model with one graph break, and
the speedup came out at or below 1x. Explain the mechanism: what does Dynamo do
at a break, and why does `n` breaks mean `n+1` graphs? What did `.item()`
specifically force the runtime to do that a pure-tensor op would not?

*Your reasoning here (ungraded - for your own learning).*

---

### 2. Which bug dominated the numerical divergence, and why not the other two?

Part 3.1 measured a gap of order `1e-4` between the buggy and fixed models.
Three bugs were fixed, but they do not contribute equally. Work out which one
dominates - and for each of the other two, say why its contribution is small
*on this input, in fp32*. (One of them does not fire at all: check the value of
`attn.max()` against `scale_threshold`.) Then predict what changes in fp16.

*Your reasoning here (ungraded - for your own learning).*

---

### 3. The tolerance question.

Your bisection found stages 1 and 2 matching eager **exactly** and stage 3
differing by a fraction of one fp32 ulp. Suppose a colleague proposes
`atol=1e-6` as the project's standard for "compiled matches eager".

Using your own numbers: what verdict would that tolerance return for stage 3,
and is that verdict correct? Then state the general rule - what must a
tolerance be expressed *relative to* if it is to mean anything? Why is
comparing eager against *itself* first a better starting point than picking a
constant?

*Your reasoning here (ungraded - for your own learning).*

---

### 4. The diagnostic funnel, on Spyre.

Classify each of the three bugs in the funnel layers from lecture: which layer
does the bug *originate* at, and which layer does its *symptom* show up at?

Then predict: on a dataflow target with **no eager fallback**, which bug has the
most severe consequence - a crash, a wrong answer, or a performance regression?
Say why the absence of a fallback is what changes the severity.

You will revisit this in Week 6. It's okay to be wrong, but be *concrete*.

*Your reasoning here (ungraded - for your own learning).*


---

## Part 7: Save and export your results

The cell below writes **`lab5_results.json`** and prints it. Copy its contents
into the `RESULTS_JSON` cell of the **grader notebook** on Coursera Labs and run
every cell there to see your **40 ungraded Self-Assessment points**.

It carries the **raw inputs** next to every derived number - both timings behind
each speedup, all three per-stage bisection differences, the two reordered sums -
so the grader can recompute each derived value rather than take it on trust.

> Those 40 points are **feedback, not a grade** - they tell you how well the lab
> went. The graded item for this week is the **Concept Check quiz** (10 points,
> auto-graded on Coursera, 80% to pass).


In [ ]:
# === Part 7: Save your results for self-assessment =====================
# This writes lab5_results.json. Paste its contents into the grader notebook
# on Coursera Labs to see your 40 ungraded Self-Assessment points.
#
# PROVIDED - do not edit. Everything here is built from variables you already
# computed in Parts 1-6, so there is nothing to fill in. Just run it.
import json as _json

RESULTS = {
    # --- Part 1: the breaks, and what they cost --------------------------
    # graph_count must equal graph_break_count + 1; the grader rechecks that.
    "graph_breaks": {
        "count":         int(graph_break_count),
        "graph_count":   int(graph_count),
        "break_reasons": [str(r)[:300] for r in break_reasons],
    },
    # Both timings travel with the speedup so the grader can redo the division.
    "timing_buggy": {
        "eager_ms":    float(eager_ms),
        "compiled_ms": float(compiled_ms),
        "speedup":     float(buggy_speedup),
    },

    # --- Part 2: the fix ------------------------------------------------
    "fixed_model": {
        "break_count":       int(fixed_break_count),
        "graph_count":       int(fixed_graph_count),
        "compiled_ms":       float(fixed_compiled_ms),
        "speedup_vs_buggy":  float(fix_speedup),
    },

    # --- Part 3: numerical divergence, model fix vs compiler noise ------
    "divergence": {
        "buggy_vs_fixed_max_abs":    float(max_abs_diff),
        "buggy_vs_fixed_mean_abs":   float(mean_abs_diff),
        "buggy_vs_fixed_max_rel":    float(max_rel_diff),
        "compilation_only_max_abs":  float(compilation_max_abs),
        "compilation_only_mean_abs": float(compilation_mean_abs),
        "compilation_only_max_rel":  float(compilation_max_rel),
        "is_bug2_divergence_larger": bool(comparison["is_bug2_divergence_larger"]),
        "ratio":                     float(comparison["ratio"]),
    },

    # --- Part 3b: tolerance selection. Pure arithmetic, no hardware -----
    "tolerance": {
        "sum_fwd":     tol_sum_fwd,
        "sum_rev":     tol_sum_rev,
        "abs_diff":    tol_abs_diff,
        "atol_tight":  TOL_TIGHT,
        "atol_loose":  TOL_LOOSE,
        "pass_tight":  tol_pass_tight,
        "pass_loose":  tol_pass_loose,
    },

    # --- Part 4: bisection. All three stages travel, so the grader can
    #     rederive first_divergent_stage instead of trusting it. ---------
    "bisection": {
        "stage1_max_abs":        float(bisection_result["stage1_max_abs"]),
        "stage2_max_abs":        float(bisection_result["stage2_max_abs"]),
        "stage3_max_abs":        float(bisection_result["stage3_max_abs"]),
        # Eager vs itself: the yardstick the stage verdict is derived against,
        # so the grader can redo that derivation without a magic constant.
        "eager_self_diff":       float(bisection_result["eager_self_diff"]),
        "output_scale":          float(bisection_result["output_scale"]),
        "fp32_ulp":              float(bisection_result["fp32_ulp"]),
        "stage3_ulps":           float(bisection_result["stage3_ulps"]),
        "first_divergent_stage": int(bisection_result["first_divergent_stage"]),
        "is_this_a_bug":         bool(bisection_result["is_this_a_bug"]),
        "divergence_cause":      str(bisection_result["divergence_cause"])[:500],
    },

    # --- Part 5: what the compiler actually emitted ---------------------
    "artifacts": {
        "fx_n_ops":       int(fx_n_ops),
        "fx_n_matmul":    int(fx_n_matmul),
        "triton_kernels": int(triton_kernels),
        "extern_kernels": int(extern_kernels),
    },

    # --- Part 6: your Landscape Map row --------------------------------
    "landscape_row_w5": landscape_row_w5,

    # --- environment: reported, never graded against -------------------
    "environment": {
        "device": (torch.cuda.get_device_name(0)
                   if torch.cuda.is_available() else "CPU only"),
        "torch_version": torch.__version__,
        "mode": MODE,
    },
}

with open("lab5_results.json", "w") as _f:
    _json.dump(RESULTS, _f, indent=2)

print("Saved lab5_results.json\n")
print(_json.dumps(RESULTS, indent=2))
print("\n" + "=" * 70)
print("NEXT STEPS")
print("=" * 70)
print("1. Copy the JSON printed above (from the opening '{' through the")
print("   final '}'), or open the file browser (folder icon, left sidebar)")
print("   and double-click lab5_results.json.")
print("2. Open the grader notebook on Coursera Labs.")
print("3. Paste the contents into the RESULTS_JSON cell, then run every cell.")
print("   It reports your 40 ungraded Self-Assessment points.")
print("4. The Concept Check quiz is a SEPARATE auto-graded Coursera item")
print("   worth 10 graded points. It is mandatory and needs 80% to pass.")
print("\nKeep this Colab notebook: if a check fails you will want to re-run")
print("the cell that produced the value.")

# Optional: download the file instead of copy-pasting.
# from google.colab import files
# files.download('lab5_results.json')


---
## ★ Optional: Dataflow Deep Dive (Spyre hardware)
_Requires Spyre access via an IBM Research partnership. Skip on the core track._

1. Attempt to compile the `BuggyTransformerBlock` on Spyre (`torch_sendnn`).
2. Document which bugs become **compilation errors** (Spyre rejects) vs **runtime errors**.
3. Use `SENDNN_LOG_LEVEL=DEBUG` to capture the Spyre compiler's IR and identify where
   each bug manifests in the Spyre compilation pipeline.
4. Compare: on GPU, Bug 1 is a silent perf regression. On Spyre, does it compile at all?

> **Spyre vocabulary (consistent across all labs):** `import torch_sendnn` registers the `aiu` device; compile with `torch.compile(model, backend="sendnn")`; dump the backend IR with `TORCH_COMPILE_DEBUG=1` (look for `sdsc_*.json`); raise verbosity with `SENDNN_LOG_LEVEL=DEBUG`. Requires Spyre access via an IBM Research partnership.


In [ ]:
# Spyre track only, pseudocode skeleton
# import torch_sendnn
# model_spyre = BuggyTransformerBlock(HIDDEN_DIM, NUM_HEADS, FFN_DIM).to('aiu').eval()
# try:
#     compiled_spyre = torch.compile(model_spyre, backend='sendnn')
#     out = compiled_spyre(dummy.to('aiu'))
# except Exception as e:
#     print(f"Spyre compilation failed: {e}")
#     # Which bug caused this? Document it.

---

### Wrap-up checklist
- [ ] **Runtime -> Change runtime type -> T4 GPU** set before running anything.
- [ ] Runtime -> Run all; every **Checkpoint** prints `[OK]`.
- [ ] Part 1: graph breaks detected and counted; eager vs compiled benchmarked.
- [ ] Part 2: all three bugs fixed; `dynamo.explain` reports 0 breaks, 1 graph.
- [ ] Part 3: divergence quantified for both comparisons.
- [ ] Part 4: bisection run; `first_divergent_stage` **derived** from your measurements.
- [ ] Part 5: debug artifacts inspected; Triton and extern kernels counted.
- [ ] Part 6: Landscape Map row complete and consistent.
- [ ] Part 7 run, `lab5_results.json` written and copied.
- [ ] JSON pasted into the grader notebook on Coursera, which reports your 40
      ungraded Self-Assessment points.
- [ ] **Concept Check quiz** taken - a separate auto-graded Coursera Graded
      Assignment worth **10 graded points**. Mandatory, 80% to pass.

_End of Lab 5. Next week: Roofline Analysis, quantifying whether your bottleneck is fixable._

---

*Course: Dataflow Architectures for AI, Week 5 Lab, v1.0 · © 2026 Dr. Kaoutar El Maghraoui*
